# Intent Classification EDA and Baseline

This notebook explores **intent classification** for the Jarvis Voice Agent.

Goals:
- Understand the kinds of user utterances Jarvis should handle (weather, reminders, notes, general Q&A).
- Inspect label distribution and text characteristics.
- Build a simple but strong baseline: TF-IDF + linear classifier.
- Choose metrics and evaluation strategy (accuracy, macro F1, confusion matrix).
- Use feature importance to understand what the model is learning.

The analysis is designed to run **offline** using `data/sample_intents.csv`, but can optionally
use a larger dataset derived from the **DeepPavlov/clinc_oos** dataset if downloaded via
`scripts/download_intent_dataset.py`.

## 1. Setup

We import core libraries for data analysis and model training.

- `pandas` for tabular data.
- `matplotlib` / `seaborn` for plots.
- `scikit-learn` for vectorization, models, and metrics.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

sns.set(style="whitegrid")

## 2. Load Dataset

We follow a **fallback strategy**:

1. If `data/clinc_oos_intents.csv` exists (downloaded from the CLINC OOS dataset), we use it.
2. Otherwise, we fall back to `data/sample_intents.csv`, a small offline-friendly sample.

Both datasets have the same columns:
- `text`: the user utterance.
- `intent`: the target label.
- `split`: which split the example belongs to (`train`, `validation`, `test`).

In [ ]:
data_dir = Path("data")
full_path = data_dir / "clinc_oos_intents.csv"
sample_path = data_dir / "sample_intents.csv"

if full_path.exists():
    print(f"Loading full dataset from {full_path}")
    df = pd.read_csv(full_path)
else:
    print(f"Loading sample dataset from {sample_path}")
    df = pd.read_csv(sample_path)

df.head()

## 3. Dataset Overview

We inspect basic properties:

- Number of examples.
- Number of unique intents.
- Distribution of examples across splits.
- Label distribution (class balance).

In [ ]:
print("Total examples:", len(df))
print("Unique intents:", df["intent"].nunique())
print("Splits:")
print(df["split"].value_counts())

df["intent"].value_counts().plot(kind="bar")
plt.title("Intent distribution")
plt.xlabel("Intent")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Class Imbalance

For a production assistant, we care about **macro F1** in addition to accuracy:

- Accuracy is dominated by frequent intents.
- Macro F1 treats each intent equally and reveals how we perform on rare classes.

In our early dataset, we expect some imbalance (e.g., more general Q&A than rare commands).

## 4. Text Length and Token Analysis

Understanding typical utterance length helps set reasonable limits for:
- Backend schema (`MAX_CONTENT_LENGTH`).
- Truncation strategies if we move to on-device models.

We compute basic length statistics and visualize distributions.

In [ ]:
df["char_len"] = df["text"].str.len()
df["word_len"] = df["text"].str.split().str.len()

print(df[["char_len", "word_len"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["char_len"], bins=20, ax=axes[0])
axes[0].set_title("Character length distribution")

sns.histplot(df["word_len"], bins=15, ax=axes[1])
axes[1].set_title("Word length distribution")
plt.tight_layout()
plt.show()

### Preprocessing Decisions

For a simple baseline we use minimal preprocessing:

- Lowercasing text.
- Letting `TfidfVectorizer` handle tokenization and stopwords.

We avoid heavy normalization (e.g., stemming) initially, to keep the model and debugging simple.
If we see systematic errors later (e.g., case sensitivity, misspellings), we can revisit this.

In [ ]:
df["text_norm"] = df["text"].str.lower()
df[["text", "text_norm"]].head()

## 5. Train / Validation / Test Split

If the dataset already has a `split` column (as in our case), we respect it.
Otherwise, we would create a stratified split. Using explicit splits helps avoid information leakage
between training and evaluation.

In [ ]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print(len(train_df), len(val_df), len(test_df))

## 6. Baseline Model: TF-IDF + Logistic Regression

We use a simple and strong baseline:

- Vectorizer: `TfidfVectorizer` with word-level tokens.
- Classifier: `LogisticRegression` with a multinomial setting.

This gives us a fast, interpretable model with decent performance, and is easy to deploy in Python.
For on-device NLU later, we may replace this with a distilled transformer or on-device model,
but the baseline defines a **minimum acceptable quality bar**.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train = vectorizer.fit_transform(train_df["text_norm"])
y_train = train_df["intent"]

X_val = vectorizer.transform(val_df["text_norm"])
y_val = val_df["intent"]

X_test = vectorizer.transform(test_df["text_norm"])
y_test = test_df["intent"]

clf = LogisticRegression(max_iter=1000, multi_class="multinomial")
clf.fit(X_train, y_train)

y_pred_val = clf.predict(X_val)
y_pred_test = clf.predict(X_test)

val_acc = accuracy_score(y_val, y_pred_val)
val_f1 = f1_score(y_val, y_pred_val, average="macro")
test_acc = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test, average="macro")

print(f"Validation accuracy: {val_acc:.3f}, macro F1: {val_f1:.3f}")
print(f"Test accuracy: {test_acc:.3f}, macro F1: {test_f1:.3f}")

### Why Accuracy and Macro F1?

- **Accuracy**: easy to interpret, but can hide poor performance on rare intents.
- **Macro F1**: averages F1 across all intents, giving equal weight to each.

For Jarvis, we care about **not missing important intents** (e.g., reminders). A model that is good at
general Q&A but consistently fails on reminders is not acceptable, even if its overall accuracy looks high.
Macro F1 surfaces this issue.

## 7. Confusion Matrix and Error Analysis

We look at the confusion matrix to understand which intents the model confuses most often.
This directly informs both data collection and UX design (e.g., when to ask clarifying questions).

In [ ]:
labels = sorted(df["intent"].unique())
cm = confusion_matrix(y_test, y_pred_test, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.ylabel("True intent")
plt.xlabel("Predicted intent")
plt.title("Confusion matrix (test)")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_test))

From this matrix, we can identify patterns such as:

- `weather_query` being misclassified as `general_qa` when the utterance is vague.
- `set_reminder` vs `save_note` confusion when the language is ambiguous.

These insights can guide prompt design and tool routing:
- For ambiguous cases, prefer a **clarifying question** before committing to a tool.
- Consider explicit keywords ("remind", "note", "save", "weather", "forecast") as features in
  future rule-augmented models.

## 8. Feature Importance

Logistic regression coefficients provide a simple notion of feature importance:
- Positive weights for a given intent highlight n-grams strongly associated with that intent.

We inspect the top features per intent to verify that the model's behavior aligns with our intuition
(e.g., "weather" and "rain" for `weather_query`, "remind" and "tomorrow" for `set_reminder`).

In [ ]:
feature_names = vectorizer.get_feature_names_out()
classes = clf.classes_
coefs = clf.coef_

def top_features_for_class(class_index: int, top_k: int = 10):
    coef = coefs[class_index]
    top_idx = coef.argsort()[::-1][:top_k]
    return list(zip(feature_names[top_idx], coef[top_idx]))

for idx, intent in enumerate(classes):
    print(f"\nTop features for intent='{intent}':")
    for feat, weight in top_features_for_class(idx, top_k=8):
        print(f"  {feat:20s} {weight: .3f}")

If the top features look unintuitive or dominated by artifacts (e.g., punctuation), we would:

- Adjust preprocessing (e.g., strip punctuation, tune n-gram range).
- Collect more diverse examples for underrepresented intents.

For a small dataset, we expect some noise, but even then this inspection helps us calibrate whether
the model is learning sensible patterns.

## 9. Design Decisions for Jarvis NLU

Based on this baseline analysis:

1. **Metrics**
   - Track both accuracy and macro F1, with macro F1 as the primary metric for intent quality.
   - For production, we would set minimal thresholds per key intent (e.g., reminders & weather).

2. **Baseline model**
   - TF-IDF + Logistic Regression is:
     - Lightweight.
     - Interpretable.
     - Easy to deploy in Python.
   - It defines a floor: on-device or transformer-based models should match or exceed this performance.

3. **Data needs**
   - If confusion between `set_reminder` and `save_note` is high, we should collect more
     examples of those intents and use clearer language in prompts.
   - For `weather_query`, ensure coverage of different ways users ask about weather ("rain",
     "temperature", "forecast", etc.).

4. **Integration with tool routing**
   - The same intent taxonomy used here (weather, reminders, notes, general QA) can drive
     the tool router system prompt.
   - As we move toward on-device NLU, we can:
     - Train a small intent classifier using this dataset.
     - Use its predictions as an additional signal alongside LLM tool routing.

This notebook should evolve along with the project: as we add new tools and intents (e.g., timers,
smart home commands), we will extend the dataset, update the taxonomy, and re-run this analysis.